# Sephora Moisturizer Customer Insights Pipeline

## 01 — Data Audit and Preparation

### Objective

This notebook validates the source files, audits data quality, loads
the raw files into SQLite staging tables, and confirms the final
analysis cohort:

- Primary category: Skincare
- Secondary category: Moisturizers
- Tertiary category: Moisturizers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

data_dir = Path(
    "/content/drive/MyDrive/sephora dataset"
)

db_path = data_dir / "sephora_moisturizer.db"

In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive/sephora dataset")

found_review_files = sorted(
    drive_root.rglob("reviews_*.csv")
)

print("Review files found:", len(found_review_files))

for file in found_review_files:
    print(file)

Review files found: 5
/content/drive/MyDrive/sephora dataset/reviews_0-250_masked.csv
/content/drive/MyDrive/sephora dataset/reviews_1250-end_masked.csv
/content/drive/MyDrive/sephora dataset/reviews_250-500_masked.csv
/content/drive/MyDrive/sephora dataset/reviews_500-750_masked.csv
/content/drive/MyDrive/sephora dataset/reviews_750-1250_masked.csv


In [ ]:
data_dir = drive_root
review_files = found_review_files
product_file = data_dir / "product_info.csv"

print("Data directory:", data_dir)
print("Review files:", len(review_files))
print("Product file exists:", product_file.exists())

if len(review_files) != 5:
    raise ValueError(
        f"Expected 5 review files, found {len(review_files)}"
    )

if not product_file.exists():
    raise FileNotFoundError(
        f"Missing product file: {product_file}"
    )

print("\nFile validation passed.")

Data directory: /content/drive/MyDrive/sephora dataset
Review files: 5
Product file exists: True

File validation passed.


In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

# Save the SQLite database in Google Drive
database_dir = Path(
    "/content/drive/MyDrive/sephora-moisturizer-insights"
)
database_dir.mkdir(parents=True, exist_ok=True)

database_path = database_dir / "sephora_reviews.db"

conn = sqlite3.connect(database_path)

import_summary = []

# Import five review files as separate staging tables
for index, file in enumerate(review_files, start=1):
    table_name = f"reviews_part_{index}"

    review_part = pd.read_csv(
        file,
        low_memory=False
    )

    # Retain source-file information for traceability
    review_part["source_file"] = file.name

    review_part.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    import_summary.append({
        "table_name": table_name,
        "source_file": file.name,
        "rows": len(review_part),
        "columns": len(review_part.columns)
    })

    print(
        f"Imported {file.name}: "
        f"{len(review_part):,} rows → {table_name}"
    )

# Import product catalog
products = pd.read_csv(
    product_file,
    low_memory=False
)

products.to_sql(
    "products_raw",
    conn,
    if_exists="replace",
    index=False
)

print(
    f"\nImported product_info.csv: "
    f"{len(products):,} rows → products_raw"
)

display(pd.DataFrame(import_summary))
print("\nDatabase saved to:", database_path)

Imported reviews_0-250_masked.csv: 168,432 rows → reviews_part_1
Imported reviews_1250-end_masked.csv: 10,805 rows → reviews_part_2
Imported reviews_250-500_masked.csv: 51,852 rows → reviews_part_3
Imported reviews_500-750_masked.csv: 29,283 rows → reviews_part_4
Imported reviews_750-1250_masked.csv: 25,040 rows → reviews_part_5

Imported product_info.csv: 8,494 rows → products_raw


,table_name,source_file,rows,columns
0,reviews_part_1,reviews_0-250_masked.csv,168432,20
1,reviews_part_2,reviews_1250-end_masked.csv,10805,20
2,reviews_part_3,reviews_250-500_masked.csv,51852,20
3,reviews_part_4,reviews_500-750_masked.csv,29283,20
4,reviews_part_5,reviews_750-1250_masked.csv,25040,20



Database saved to: /content/drive/MyDrive/sephora-moisturizer-insights/sephora_reviews.db


## 4. Combine Review Tables with SQL

The five source review files are stored as separate SQLite staging tables.  
`UNION ALL` is used to preserve every source row before duplicate and quality checks.

In [ ]:
# Combine all five review staging tables with SQL
conn.executescript("""
DROP TABLE IF EXISTS reviews_all;

CREATE TABLE reviews_all AS

SELECT * FROM reviews_part_1
UNION ALL
SELECT * FROM reviews_part_2
UNION ALL
SELECT * FROM reviews_part_3
UNION ALL
SELECT * FROM reviews_part_4
UNION ALL
SELECT * FROM reviews_part_5;
""")

# Validate the combined table
review_audit = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS unique_products,
    MIN(rating) AS min_rating,
    MAX(rating) AS max_rating,
    SUM(
        CASE WHEN product_id IS NULL THEN 1 ELSE 0 END
    ) AS missing_product_id,
    SUM(
        CASE WHEN review_text IS NULL
                  OR TRIM(review_text) = ''
             THEN 1 ELSE 0 END
    ) AS missing_review_text,
    MIN(submission_time) AS earliest_review,
    MAX(submission_time) AS latest_review
FROM reviews_all;
""", conn)

display(review_audit)

,total_rows,unique_products,min_rating,max_rating,missing_product_id,missing_review_text,earliest_review,latest_review
0,285412,541,1,5,0,345,2008-08-28,2023-03-21


In [ ]:
source_audit = pd.read_sql_query("""
SELECT
    source_file,
    COUNT(*) AS row_count,
    COUNT(DISTINCT product_id) AS unique_products
FROM reviews_all
GROUP BY source_file
ORDER BY source_file;
""", conn)

display(source_audit)

print(
    "Row-count validation:",
    source_audit["row_count"].sum()
)

assert source_audit["row_count"].sum() == 285412

print("Review table validation passed.")

,source_file,row_count,unique_products
0,reviews_0-250_masked.csv,168432,61
1,reviews_1250-end_masked.csv,10805,252
2,reviews_250-500_masked.csv,51852,62
3,reviews_500-750_masked.csv,29283,62
4,reviews_750-1250_masked.csv,25040,104


Row-count validation: 285412
Review table validation passed.


In [ ]:
product_audit = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_product_rows,
    COUNT(DISTINCT product_id) AS unique_product_ids,
    SUM(
        CASE WHEN product_id IS NULL THEN 1 ELSE 0 END
    ) AS missing_product_ids
FROM products_raw;
""", conn)

display(product_audit)

,total_product_rows,unique_product_ids,missing_product_ids
0,8494,8494,0


## 5. Build the Moisturizer Analysis Cohort

Reviews are joined with the product catalog using `product_id`. The final
cohort includes only products classified as Skincare > Moisturizers >
Moisturizers.

Two separate negative-review indicators are created:

- `is_negative`: rating is 1 or 2 stars
- `eligible_for_llm`: negative review with more than 20 characters of text

In [ ]:
# Add indexes to improve JOIN performance
conn.executescript("""
CREATE INDEX IF NOT EXISTS idx_reviews_product_id
ON reviews_all(product_id);

CREATE INDEX IF NOT EXISTS idx_products_product_id
ON products_raw(product_id);
""")

# Create final moisturizer analysis table
conn.executescript("""
DROP TABLE IF EXISTS moisturizer_reviews;

CREATE TABLE moisturizer_reviews AS
SELECT
    r.*,

    p.product_name AS catalog_product_name,
    p.brand_name AS catalog_brand_name,
    p.price_usd AS catalog_price_usd,
    p.primary_category,
    p.secondary_category,
    p.tertiary_category,

    CASE
        WHEN r.rating <= 2 THEN 1
        ELSE 0
    END AS is_negative,

    CASE
        WHEN r.rating <= 2
             AND r.review_text IS NOT NULL
             AND LENGTH(TRIM(r.review_text)) > 20
        THEN 1
        ELSE 0
    END AS eligible_for_llm

FROM reviews_all AS r

INNER JOIN products_raw AS p
    ON r.product_id = p.product_id

WHERE p.primary_category = 'Skincare'
  AND p.secondary_category = 'Moisturizers'
  AND p.tertiary_category = 'Moisturizers';
""")

conn.commit()

In [ ]:
cohort_audit = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT product_id) AS unique_products,
    COUNT(DISTINCT catalog_brand_name) AS unique_brands,

    ROUND(AVG(rating), 3) AS avg_rating,

    SUM(is_negative) AS negative_reviews,
    ROUND(
        AVG(CAST(is_negative AS REAL)),
        4
    ) AS negative_rate,

    SUM(eligible_for_llm) AS llm_eligible_reviews,

    SUM(
        CASE WHEN review_text IS NULL
                  OR TRIM(review_text) = ''
             THEN 1 ELSE 0 END
    ) AS missing_review_text

FROM moisturizer_reviews;
""", conn)

display(cohort_audit)

,total_reviews,unique_products,unique_brands,avg_rating,negative_reviews,negative_rate,llm_eligible_reviews,missing_review_text
0,39541,71,46,4.404,3023,0.0765,3022,23


In [ ]:
category_check = pd.read_sql_query("""
SELECT
    primary_category,
    secondary_category,
    tertiary_category,
    COUNT(*) AS review_count,
    COUNT(DISTINCT product_id) AS product_count
FROM moisturizer_reviews
GROUP BY
    primary_category,
    secondary_category,
    tertiary_category;
""", conn)

display(category_check)

,primary_category,secondary_category,tertiary_category,review_count,product_count
0,Skincare,Moisturizers,Moisturizers,39541,71


## 6. Pre-Deduplication Data-Quality Audit
Missing values, potential duplicate reviews, rating distributions, and
segmentation-field coverage are checked before business analysis.

In [ ]:
missingness_audit = pd.read_sql_query("""
SELECT
    'product_id' AS field,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_count,
    ROUND(
        100.0 * SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS missing_pct
FROM moisturizer_reviews

UNION ALL

SELECT
    'rating',
    SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END),
    ROUND(
        100.0 * SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    )
FROM moisturizer_reviews

UNION ALL

SELECT
    'review_text',
    SUM(
        CASE WHEN review_text IS NULL
                  OR TRIM(review_text) = ''
             THEN 1 ELSE 0 END
    ),
    ROUND(
        100.0 * SUM(
            CASE WHEN review_text IS NULL
                      OR TRIM(review_text) = ''
                 THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    )
FROM moisturizer_reviews

UNION ALL

SELECT
    'skin_type',
    SUM(
        CASE WHEN skin_type IS NULL
                  OR TRIM(skin_type) = ''
             THEN 1 ELSE 0 END
    ),
    ROUND(
        100.0 * SUM(
            CASE WHEN skin_type IS NULL
                      OR TRIM(skin_type) = ''
                 THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    )
FROM moisturizer_reviews

UNION ALL

SELECT
    'is_recommended',
    SUM(CASE WHEN is_recommended IS NULL THEN 1 ELSE 0 END),
    ROUND(
        100.0 * SUM(
            CASE WHEN is_recommended IS NULL THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    )
FROM moisturizer_reviews

UNION ALL

SELECT
    'catalog_price_usd',
    SUM(CASE WHEN catalog_price_usd IS NULL THEN 1 ELSE 0 END),
    ROUND(
        100.0 * SUM(
            CASE WHEN catalog_price_usd IS NULL THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    )
FROM moisturizer_reviews

UNION ALL

SELECT
    'submission_time',
    SUM(CASE WHEN submission_time IS NULL THEN 1 ELSE 0 END),
    ROUND(
        100.0 * SUM(
            CASE WHEN submission_time IS NULL THEN 1 ELSE 0 END
        ) / COUNT(*),
        2
    )
FROM moisturizer_reviews;
""", conn)

display(missingness_audit)

,field,missing_count,missing_pct
0,product_id,0,0.00
1,rating,0,0.00
2,review_text,23,0.06
3,skin_type,3824,9.67
4,is_recommended,9122,23.07
5,catalog_price_usd,0,0.00
6,submission_time,0,0.00


### Missing-Value Treatment

- Product ID, rating, catalog price, and submission date were complete.
- Review text was missing in 23 records (0.06%). These records were retained
  for structured analysis but excluded from LLM classification when the text
  was insufficient.
- Skin type was missing in 3,824 records (9.67%). No skin type was imputed.
  Skin-type comparisons will use the 35,717 reviews with a reported value.
- Recommendation status was missing in 9,122 records (23.07%). Missing values
  were not treated as non-recommendations. Recommendation rates will use the
  30,419 reviews with a recorded response.
- Skin type is self-reported and unavailable for some reviewers, so
  segmentation findings may not fully represent all moisturizer customers.

## 7. Potential Duplicate Review Audit

The masked dataset does not include a reviewer identifier. Potential duplicate
reviews are therefore identified using product ID, submission date, rating,
and exact review text. Matching records are treated as potential rather than
confirmed duplicates and are inspected before any removal.

In [ ]:
column_audit = pd.read_sql_query("""
PRAGMA table_info(moisturizer_reviews);
""", conn)

display(column_audit[["name", "type"]])

,name,type
0,Unnamed: 0.1,INT
1,Unnamed: 0,INT
2,rating,INT
3,is_recommended,REAL
4,helpfulness,REAL
5,total_feedback_count,INT
6,total_neg_feedback_count,INT
7,total_pos_feedback_count,INT
8,submission_time,TEXT
9,review_text,TEXT


In [ ]:
duplicate_audit = pd.read_sql_query("""
WITH potential_duplicate_groups AS (
    SELECT
        product_id,
        submission_time,
        rating,
        review_text,
        COUNT(*) AS occurrences
    FROM moisturizer_reviews
    WHERE review_text IS NOT NULL
      AND TRIM(review_text) <> ''
    GROUP BY
        product_id,
        submission_time,
        rating,
        review_text
    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS duplicate_groups,
    COALESCE(
        SUM(occurrences) - COUNT(*),
        0
    ) AS potential_extra_rows
FROM potential_duplicate_groups;
""", conn)

display(duplicate_audit)

,duplicate_groups,potential_extra_rows
0,20,26


In [ ]:
duplicate_examples = pd.read_sql_query("""
SELECT
    product_id,
    catalog_product_name,
    submission_time,
    rating,
    review_text,
    COUNT(*) AS occurrences
FROM moisturizer_reviews
WHERE review_text IS NOT NULL
  AND TRIM(review_text) <> ''
GROUP BY
    product_id,
    catalog_product_name,
    submission_time,
    rating,
    review_text
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
LIMIT 10;
""", conn)

display(duplicate_examples)

,product_id,catalog_product_name,submission_time,rating,review_text,occurrences
0,P248407,Ultra Repair Cream Intense Hydration,2014-05-03,5,"First of, I have and prove skin & I started us...",8
1,P122900,Dramatically Different Moisturizing Gel,2019-09-06,1,received a trial size recently and it destroye...,2
2,P122900,Dramatically Different Moisturizing Gel,2020-12-15,5,"I tent to buy Mini size to try out, and im bey...",2
3,P248407,Ultra Repair Cream Intense Hydration,2018-12-31,5,Really good stuff—I had gotten this as a cheap...,2
4,P248407,Ultra Repair Cream Intense Hydration,2020-02-26,5,Don’t judge this product over its price!!! I s...,2
5,P248407,Ultra Repair Cream Intense Hydration,2021-01-04,4,This product truly deserves its accolades! Wh...,2
6,P248407,Ultra Repair Cream Intense Hydration,2021-01-09,5,I’ll admit it – I’m a bit skeptical of product...,2
7,P248407,Ultra Repair Cream Intense Hydration,2021-01-13,4,My go-to lotion for my hands. I have 8oz tubes...,2
8,P416561,Squalane + Probiotic Balancing Gel Moisturizer,2020-12-26,3,I love the smell and the concept of this produ...,2
9,P430337,C-Rush Vitamin C Gel Moisturizer,2018-04-16,5,Ole Henriksen hasn’t disappointed yet. I am ab...,2


In [ ]:
duplicate_consistency = pd.read_sql_query("""
WITH duplicate_groups AS (
    SELECT
        product_id,
        catalog_product_name,
        submission_time,
        rating,
        review_text,
        COUNT(*) AS occurrences,

        COUNT(
            DISTINCT COALESCE(skin_type, '<NULL>')
        ) AS skin_type_versions,

        COUNT(
            DISTINCT COALESCE(
                CAST(is_recommended AS TEXT),
                '<NULL>'
            )
        ) AS recommendation_versions,

        GROUP_CONCAT(
            DISTINCT source_file
        ) AS source_files

    FROM moisturizer_reviews

    WHERE review_text IS NOT NULL
      AND TRIM(review_text) <> ''

    GROUP BY
        product_id,
        catalog_product_name,
        submission_time,
        rating,
        review_text

    HAVING COUNT(*) > 1
)

SELECT
    product_id,
    catalog_product_name,
    submission_time,
    rating,
    occurrences,
    skin_type_versions,
    recommendation_versions,
    source_files
FROM duplicate_groups
ORDER BY occurrences DESC;
""", conn)

display(duplicate_consistency)

,product_id,catalog_product_name,submission_time,rating,occurrences,skin_type_versions,recommendation_versions,source_files
0,P248407,Ultra Repair Cream Intense Hydration,2014-05-03,5,8,1,1,reviews_0-250_masked.csv
1,P122900,Dramatically Different Moisturizing Gel,2019-09-06,1,2,1,1,reviews_0-250_masked.csv
2,P122900,Dramatically Different Moisturizing Gel,2020-12-15,5,2,1,1,reviews_0-250_masked.csv
3,P248407,Ultra Repair Cream Intense Hydration,2018-12-31,5,2,1,1,reviews_0-250_masked.csv
4,P248407,Ultra Repair Cream Intense Hydration,2020-02-26,5,2,1,1,reviews_0-250_masked.csv
5,P248407,Ultra Repair Cream Intense Hydration,2021-01-04,4,2,1,1,reviews_0-250_masked.csv
6,P248407,Ultra Repair Cream Intense Hydration,2021-01-09,5,2,1,1,reviews_0-250_masked.csv
7,P248407,Ultra Repair Cream Intense Hydration,2021-01-13,4,2,1,1,reviews_0-250_masked.csv
8,P416561,Squalane + Probiotic Balancing Gel Moisturizer,2020-12-26,3,2,1,1,reviews_250-500_masked.csv
9,P430337,C-Rush Vitamin C Gel Moisturizer,2018-04-16,5,2,1,1,reviews_0-250_masked.csv


In [ ]:
# Get all columns in the cohort table
table_columns = pd.read_sql_query("""
PRAGMA table_info(moisturizer_reviews);
""", conn)["name"].tolist()

# Exclude technical source/index fields from exact-duplicate comparison
excluded_columns = [
    column
    for column in table_columns
    if column == "source_file"
    or column.lower().startswith("unnamed")
]

dedupe_columns = [
    column
    for column in table_columns
    if column not in excluded_columns
]

partition_columns = ",\n".join(
    f'"{column}"'
    for column in dedupe_columns
)

dedupe_sql = f"""
DROP TABLE IF EXISTS moisturizer_reviews_clean;

CREATE TABLE moisturizer_reviews_clean AS
WITH ranked_reviews AS (
    SELECT
        rowid AS original_rowid,

        ROW_NUMBER() OVER (
            PARTITION BY
                {partition_columns}
            ORDER BY rowid
        ) AS duplicate_rank

    FROM moisturizer_reviews
)

SELECT original.*
FROM moisturizer_reviews AS original

INNER JOIN ranked_reviews AS ranked
    ON original.rowid = ranked.original_rowid

WHERE ranked.duplicate_rank = 1;
"""

conn.executescript(dedupe_sql)
conn.commit()

In [ ]:
dedupe_result = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*)
     FROM moisturizer_reviews) AS original_rows,

    (SELECT COUNT(*)
     FROM moisturizer_reviews_clean) AS clean_rows,

    (SELECT COUNT(*)
     FROM moisturizer_reviews)
    -
    (SELECT COUNT(*)
     FROM moisturizer_reviews_clean) AS exact_duplicates_removed;
""", conn)

display(dedupe_result)

,original_rows,clean_rows,exact_duplicates_removed
0,39541,39535,6


### Duplicate Treatment

A reduced composite key identified 20 potential duplicate groups containing
26 potential extra rows. However, several groups contained conflicting reviewer
attributes, such as different reported skin types, so they could not be safely
treated as duplicates.

A stricter exact-match comparison was therefore applied across all substantive
fields, excluding technical source and index fields. This removed 6 exact
duplicate records (approximately 0.02% of the cohort), resulting in a final
analysis table of 39,535 reviews. The original cohort was retained for
traceability.

### 8. Cross-SKU Review-Pool Deduplication

A cross-product audit found that two Fenty Skin SKUs shared all 168 review
signatures, indicating a fully syndicated review pool across product variants.
One additional review was shared between the full-size and mini versions of a
Biossance product.

To prevent duplicated customer voices from inflating brand-level sample sizes,
169 cross-SKU duplicate records were removed while retaining the canonical
full-size record. Related full-size, mini-size, and legacy SKUs were also mapped
to a common `product_family_id` for product-level analysis.

The original cleaned table was retained for traceability. All subsequent
analyses use `moisturizer_reviews_final`.

In [ ]:
# Clear any interrupted transaction
import pandas as pd
import sqlite3
from pathlib import Path

database_path = Path(
    "/content/drive/MyDrive/sephora-moisturizer-insights/sephora_reviews.db"
)

# Reuse the existing connection if it is open;
# reconnect if the runtime restarted or the connection was closed.
try:
    conn.execute("SELECT 1")
except (NameError, sqlite3.ProgrammingError):
    conn = sqlite3.connect(database_path)

conn.rollback()
# Index used to limit comparisons to the relevant product IDs
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_clean_product_id
ON moisturizer_reviews_clean(product_id);
""")

conn.executescript("""
DROP TABLE IF EXISTS cross_sku_rows_to_remove;

CREATE TEMP TABLE cross_sku_rows_to_remove AS

-- Fenty: P467250 is preferred over P476496
SELECT DISTINCT
    candidate.rowid AS clean_rowid

FROM moisturizer_reviews_clean AS candidate

INNER JOIN moisturizer_reviews_clean AS preferred
    ON preferred.product_id = 'P467250'
   AND preferred.submission_time IS candidate.submission_time
   AND preferred.rating IS candidate.rating
   AND preferred.review_title IS candidate.review_title
   AND preferred.review_text IS candidate.review_text
   AND preferred.skin_type IS candidate.skin_type
   AND preferred.is_recommended IS candidate.is_recommended

WHERE candidate.product_id = 'P476496'
  AND candidate.review_text IS NOT NULL
  AND TRIM(candidate.review_text) <> ''


UNION


-- Fenty: P467250 is preferred over P504240
SELECT DISTINCT
    candidate.rowid AS clean_rowid

FROM moisturizer_reviews_clean AS candidate

INNER JOIN moisturizer_reviews_clean AS preferred
    ON preferred.product_id = 'P467250'
   AND preferred.submission_time IS candidate.submission_time
   AND preferred.rating IS candidate.rating
   AND preferred.review_title IS candidate.review_title
   AND preferred.review_text IS candidate.review_text
   AND preferred.skin_type IS candidate.skin_type
   AND preferred.is_recommended IS candidate.is_recommended

WHERE candidate.product_id = 'P504240'
  AND candidate.review_text IS NOT NULL
  AND TRIM(candidate.review_text) <> ''


UNION


-- Fenty: P476496 is preferred over P504240
SELECT DISTINCT
    candidate.rowid AS clean_rowid

FROM moisturizer_reviews_clean AS candidate

INNER JOIN moisturizer_reviews_clean AS preferred
    ON preferred.product_id = 'P476496'
   AND preferred.submission_time IS candidate.submission_time
   AND preferred.rating IS candidate.rating
   AND preferred.review_title IS candidate.review_title
   AND preferred.review_text IS candidate.review_text
   AND preferred.skin_type IS candidate.skin_type
   AND preferred.is_recommended IS candidate.is_recommended

WHERE candidate.product_id = 'P504240'
  AND candidate.review_text IS NOT NULL
  AND TRIM(candidate.review_text) <> ''


UNION


-- Biossance: retain the full-size P433887 record
SELECT DISTINCT
    candidate.rowid AS clean_rowid

FROM moisturizer_reviews_clean AS candidate

INNER JOIN moisturizer_reviews_clean AS preferred
    ON preferred.product_id = 'P433887'
   AND preferred.submission_time IS candidate.submission_time
   AND preferred.rating IS candidate.rating
   AND preferred.review_title IS candidate.review_title
   AND preferred.review_text IS candidate.review_text
   AND preferred.skin_type IS candidate.skin_type
   AND preferred.is_recommended IS candidate.is_recommended

WHERE candidate.product_id = 'P480280'
  AND candidate.review_text IS NOT NULL
  AND TRIM(candidate.review_text) <> '';
""")


# Create the final analysis table
conn.executescript("""
DROP TABLE IF EXISTS moisturizer_reviews_final;

CREATE TABLE moisturizer_reviews_final AS

SELECT
    clean.*,

    CASE
        WHEN clean.product_id IN (
            'P467250',
            'P476496',
            'P504240'
        )
        THEN 'FENTY_INSTANT_RESET_FAMILY'

        WHEN clean.product_id IN (
            'P433887',
            'P480280'
        )
        THEN 'BIOSSANCE_OMEGA_REPAIR_FAMILY'

        ELSE clean.product_id
    END AS product_family_id,

    CASE
        WHEN clean.product_id IN (
            'P467250',
            'P476496',
            'P504240'
        )
        THEN 'Instant Reset Overnight Recovery Gel-Cream'

        WHEN clean.product_id IN (
            'P433887',
            'P480280'
        )
        THEN 'Squalane + Omega Repair Deep Hydration Moisturizer'

        ELSE clean.catalog_product_name
    END AS product_family_name

FROM moisturizer_reviews_clean AS clean

LEFT JOIN cross_sku_rows_to_remove AS removed
    ON clean.rowid = removed.clean_rowid

WHERE removed.clean_rowid IS NULL;
""")

conn.commit()


# Validate the result
cross_sku_dedupe_result = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*)
     FROM moisturizer_reviews_clean) AS clean_rows,

    (SELECT COUNT(*)
     FROM cross_sku_rows_to_remove) AS identified_duplicate_rows,

    (SELECT COUNT(*)
     FROM moisturizer_reviews_final) AS final_rows,

    (SELECT COUNT(*)
     FROM moisturizer_reviews_clean)
    -
    (SELECT COUNT(*)
     FROM moisturizer_reviews_final)
        AS cross_sku_rows_removed,

    (SELECT COUNT(DISTINCT product_id)
     FROM moisturizer_reviews_final)
        AS unique_skus,

    (SELECT COUNT(DISTINCT product_family_id)
     FROM moisturizer_reviews_final)
        AS unique_product_families;
""", conn)

display(cross_sku_dedupe_result)

,clean_rows,identified_duplicate_rows,final_rows,cross_sku_rows_removed,unique_skus,unique_product_families
0,39535,169,39366,169,70,68


In [ ]:
conn.executescript("""
CREATE INDEX IF NOT EXISTS idx_final_product_id
ON moisturizer_reviews_final(product_id);

CREATE INDEX IF NOT EXISTS idx_final_product_family_id
ON moisturizer_reviews_final(product_family_id);

CREATE INDEX IF NOT EXISTS idx_final_brand
ON moisturizer_reviews_final(catalog_brand_name);

CREATE INDEX IF NOT EXISTS idx_final_submission_time
ON moisturizer_reviews_final(submission_time);
""")

conn.commit()

In [ ]:
final_cohort_audit = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT product_id) AS unique_skus,
    COUNT(DISTINCT product_family_id) AS unique_product_families,
    COUNT(DISTINCT catalog_brand_name) AS unique_brands,

    ROUND(AVG(rating), 3) AS avg_rating,

    SUM(is_negative) AS negative_reviews,

    ROUND(
        100.0 * AVG(CAST(is_negative AS REAL)),
        2
    ) AS negative_rate_pct,

    SUM(eligible_for_llm) AS llm_eligible_reviews,

    MIN(submission_time) AS earliest_review,
    MAX(submission_time) AS latest_review

FROM moisturizer_reviews_final;
""", conn)

display(final_cohort_audit)

,total_reviews,unique_skus,unique_product_families,unique_brands,avg_rating,negative_reviews,negative_rate_pct,llm_eligible_reviews,earliest_review,latest_review
0,39366,70,68,46,4.405,2999,7.62,2998,2008-08-30,2023-03-21


In [ ]:
rating_distribution = pd.read_sql_query("""
SELECT
    rating,
    COUNT(*) AS review_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS review_pct

FROM moisturizer_reviews_final

GROUP BY rating
ORDER BY rating;
""", conn)

display(rating_distribution)

,rating,review_count,review_pct
0,1,1444,3.67
1,2,1555,3.95
2,3,2750,6.99
3,4,7477,18.99
4,5,26140,66.40


In [ ]:
skin_type_distribution = pd.read_sql_query("""
SELECT
    COALESCE(
        NULLIF(TRIM(skin_type), ''),
        'Unknown'
    ) AS skin_type,

    COUNT(*) AS review_count,

    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS review_pct

FROM moisturizer_reviews_final

GROUP BY
    COALESCE(
        NULLIF(TRIM(skin_type), ''),
        'Unknown'
    )

ORDER BY review_count DESC;
""", conn)

display(skin_type_distribution)

,skin_type,review_count,review_pct
0,combination,19447,49.40
1,dry,7288,18.51
2,oily,4508,11.45
3,normal,4306,10.94
4,Unknown,3817,9.70


## 9. Data-Audit Summary

The final analysis cohort contains 39,366 deduplicated review records covering
70 SKUs, consolidated into 68 moisturizer product families across 46 brands.

- Six exact within-SKU duplicate records were removed while the original cohort
  was retained for traceability.
- An additional 169 cross-SKU duplicate records were removed from known
  syndicated review pools involving Fenty Skin and Biossance product variants.
- Positive reviews, defined as four or five stars, accounted for 85.40% of the
  final cohort. Three-star mixed reviews accounted for 6.99%.
- The cohort contained 2,999 one- or two-star reviews, representing a 7.62%
  negative-review rate.
- Of these negative reviews, 2,998 contained sufficient text for LLM
  classification.
- Missing skin-type values were not imputed. Skin-type comparisons will use
  only reviews with a reported skin type and will emphasize within-group rates.
- The review period spans August 2008 through March 2023. Findings are
  presented as historical customer insights rather than current market
  performance.

In [ ]:
# Final completion checks
assert int(final_cohort_audit.loc[0, "total_reviews"]) == 39366
assert int(final_cohort_audit.loc[0, "unique_skus"]) == 70
assert int(
    final_cohort_audit.loc[0, "unique_product_families"]
) == 68
assert int(final_cohort_audit.loc[0, "unique_brands"]) == 46
assert int(final_cohort_audit.loc[0, "negative_reviews"]) == 2999

assert rating_distribution["review_count"].sum() == 39366
assert skin_type_distribution["review_count"].sum() == 39366

conn.commit()
conn.close()

print("01_data_audit_prep.ipynb completed successfully.")

01_data_audit_prep.ipynb completed successfully.
